---

# Data Science Competition LOGIKA UI 2026

*Prediksi risiko gagal bayar serius (DPD 90+ dalam 12 bulan) untuk pinjaman tenor 12 bulan, dinilai dengan statistik Kolmogorov-Smirnov.*

---

## DSCxxxx - lagi mager lomba bentar

- Nama Anggota 1 (Ketua)
- Nama Anggota 2
- Nama Anggota 3

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Pipeline Modules**](#3)
4. [**Feature Engineering**](#4)
5. [**Model Training**](#5)
6. [**Blending and Submission**](#6)
7. [**Conclusion**](#7)


---

# Introduction <a name="1"></a>

---

## Overview

*Lender perlu memisahkan pemohon yang akan mengalami kesulitan bayar serius dari yang tidak, memakai data yang tersedia sampai saat keputusan kredit. Data kompetisi berisi aplikasi pinjaman terpilih, profil customer, riwayat aplikasi dan cicilan sebelumnya, perilaku rekening bulanan, laporan biro kredit, dan transaksi e-commerce.*

## Aim

*Notebook ini membangun skor risiko untuk 30.000 aplikasi test dan menulis `submission.csv` (`application_id`, `default_probability`). Skor berasal dari rata-rata rank tiga keluarga model yang dipilih dengan validasi bersih (validation_v2).*

## Metric

***Kolmogorov-Smirnov (KS)** mengukur jarak maksimum antara distribusi kumulatif skor kelas gagal bayar dan kelas lancar. KS hanya bergantung pada urutan skor, jadi rata-rata rank antar model aman dipakai.*

$$\text{KS} = \max_t \left| F_1(t) - F_0(t) \right|$$

## Dataset

*Data berasal dari kompetisi Kaggle Data Science Competition LOGIKA UI 2026 (`data-science-competition-logika-ui-2026`).*

*Train berisi 70.000 aplikasi (bad rate sekitar 13,5%) dan test 30.000 aplikasi, satu aplikasi terpilih per customer, dengan periode keputusan yang sama sepanjang 2025. Semua tabel bantu sudah dipotong di `decision_datetime`; kode fitur tetap memfilter cutoff ini secara eksplisit.*

**Attributes**

1.  `train_applications` / `test_applications` : pinjaman terpilih (jumlah, bunga, cicilan, tujuan, waktu)
2.  `customer_profiles`                         : demografi dan pendapatan yang dideklarasikan
3.  `previous_applications`                     : aplikasi historis (disetujui, ditolak, dibatalkan)
4.  `repayment_history`                         : snapshot cicilan pinjaman lama pada cutoff
5.  `account_behavior`                          : arus kas rekening per bulan lengkap
6.  `bureau_observations`                       : laporan tiga biro kredit (skor, utang, inquiry)
7.  `ecommerce_transactions`                    : item belanja online (kategori, harga, status)

**Target**
 `target` : 1 jika DPD akhir hari mencapai 90 dalam 12 bulan, 0 jika tidak

## Approach: Rank Blend of Gradient Boosting and Tabular Foundation Models

*Sekitar 230 fitur level customer dibangun dari semua tabel (agregat repayment, burst belanja, premium harga per kategori, biro, arus kas). Tiga keluarga model dilatih di atasnya: enam GBDT (LightGBM, XGBoost, CatBoost, masing-masing versi dalam dan dangkal), Causilo, dan NVIDIA Kumo-Tabular large. Dua yang terakhir adalah model in-context learning pretrained yang membaca baris train sebagai context tanpa pelatihan ulang bobot.*

*GBDT dan Causilo memakai validation_v2: StratifiedKFold 5 fold, early stopping hanya di inner split 1/7 dari train fold, lalu refit di seluruh train fold dengan jumlah ronde tetap; prediksi test adalah rata-rata model fold. Kumo memakai seluruh 70.000 baris train sebagai context dan memprediksi test sekali.*

```
features (~230 kolom)
    |
    +-- 6 GBDT x 3 seed x 5 fold  -> rank rata-rata  (bobot 1/3)
    +-- Causilo x 5 fold          -> rank            (bobot 1/3)
    +-- Kumo-Tabular, context 70k -> rank            (bobot 1/3)
    |
rata-rata rank -> default_probability
```

*Di validasi bersih (OOF, seed 42) blend ini unggul +0,0022 KS atas resep blend v8 (paired bootstrap P>0 = 0,93, menang di 4 dari 5 fold).*

---

# Initialization <a name="2"></a>

---

## Environment Setup

Notebook ini dirancang untuk Kaggle dengan akselerator GPU T4 x2 dan internet aktif (unduh bobot pretrained Causilo dan Kumo dari Hugging Face). Waktu jalan tercatat sekitar 5 jam: Kumo di GPU 0 (sekitar 4,5 jam) berjalan paralel dengan GBDT dan Causilo di GPU 1 (sekitar 2 jam).

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q polars==2.0.0 lightgbm xgboost catboost structured-data-models
%pip install -q --no-deps causilo

## Import Libraries

In [ ]:
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
from scipy.stats import rankdata

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## Settings

Semua path, daftar model, dan bobot blend dipusatkan di sini. Seed outer fold untuk GBDT adalah 42, 7, dan 2026; Causilo dan Kumo memakai seed 42.

In [ ]:
class Settings:
    SEED       = 42
    _ON_KAGGLE = Path('/kaggle/input').exists()

    DATA_DIR   = next(Path('/kaggle/input' if _ON_KAGGLE else '../data').rglob('train_applications.csv')).parent
    OUTPUT_DIR = Path('/kaggle/working') if _ON_KAGGLE else Path('../results/final')
    CODE_DIR   = OUTPUT_DIR / 'pipeline'
    RUN_DIR    = OUTPUT_DIR / 'run'

    TABLES = ['train_applications', 'test_applications', 'customer_profiles', 'previous_applications',
              'repayment_history', 'account_behavior', 'bureau_observations', 'ecommerce_transactions']
    TREES  = ['lgb', 'lgb_s', 'xgb', 'xgb_s', 'cat', 'cat_s']
    TREE_SEEDS = '42,7,2026'
    ICL_SEED   = '42'

    WEIGHTS = {**{m: 1 / 18 for m in TREES}, 'causilo': 1 / 3, 'kumo_full70k': 1 / 3}

CFG = Settings()
for d in (CFG.OUTPUT_DIR, CFG.CODE_DIR, CFG.RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(CFG.DATA_DIR)

## Load Dataset

Delapan tabel dibaca dengan parsing tanggal otomatis. Tidak ada visualisasi di sini.

In [ ]:
tables = {n: pl.read_csv(CFG.DATA_DIR / f'{n}.csv', try_parse_dates=True, infer_schema_length=100000)
          for n in CFG.TABLES}
for n, t in tables.items():
    print(f'{n:24s} {t.shape}')

---

# Pipeline Modules <a name="3"></a>

---

Kode pipeline ditulis ke folder `pipeline/` dengan `%%writefile`, persis sama dengan modul yang dipakai saat validasi. `features.py` membangun fitur, `nn_prep.py` berisi transformasi rank-gaussian, `valid2.py` berisi protokol validation_v2 dan semua pembungkus model, `run_valid2.py` menjalankan model per fold dan menyimpan prediksi OOF/test per `application_id`, dan `kumo_fulltrain.py` menjalankan Kumo dengan seluruh train sebagai context. Import di dalam modul ini dieksekusi oleh subprocess, bukan oleh kernel notebook.

In [ ]:
sys.path.insert(0, str(CFG.CODE_DIR))
print(CFG.CODE_DIR)

In [ ]:
%%writefile {CFG.CODE_DIR}/features.py
"""Leak-safe, customer-level feature builders (one selected application per customer).

All auxiliary tables are already truncated at decision_datetime (verified in 01_keys_target_time.py),
but every builder still filters on the cutoff so the code stays correct if that ever changes.
"""
import polars as pl

DAY = 86400.0


def _days(a, b):
    return (pl.col(a) - pl.col(b)).dt.total_seconds() / DAY


def app_features(apps: pl.DataFrame, prof: pl.DataFrame) -> pl.DataFrame:
    d = apps.join(prof, on="customer_id", how="left")
    return d.select(
        "customer_id",
        "loan_amount", "annual_interest_rate", "proposed_monthly_installment", "loan_purpose",
        "age", "region", "employment_type", "employment_tenure_months", "declared_monthly_income",
        "dependents", "housing_status",
        decision_lag_h=(pl.col("decision_datetime") - pl.col("application_datetime")).dt.total_seconds() / 3600,
        app_hour=pl.col("application_datetime").dt.hour(),
        app_dow=pl.col("application_datetime").dt.weekday(),
        dti_declared=pl.col("proposed_monthly_installment") / pl.col("declared_monthly_income"),
        lti_declared=pl.col("loan_amount") / pl.col("declared_monthly_income"),
        tenure_over_age=pl.col("employment_tenure_months") / (pl.col("age") * 12),
        income_per_head=pl.col("declared_monthly_income") / (1 + pl.col("dependents")),
    )


def bureau_features(apps: pl.DataFrame, bureau: pl.DataFrame) -> pl.DataFrame:
    cut = apps.select("customer_id", sel_app=pl.col("application_id"), cutoff=pl.col("decision_datetime"))
    b = bureau.join(cut, on="customer_id").filter(pl.col("available_datetime") <= pl.col("cutoff"))
    b = b.with_columns(
        stale_d=_days("cutoff", "report_datetime"),
        delay_d=_days("available_datetime", "report_datetime"),
        is_sel=pl.col("application_id") == pl.col("sel_app"),
        prov=pl.col("bureau_provider").str.replace("Bureau_", "").str.to_lowercase(),
    )
    cur = b.filter("is_sel")
    num = ["score", "active_accounts", "outstanding_balance", "revolving_credit_limit",
           "revolving_outstanding_balance", "utilization", "inquiries_last_6m", "delinquent_accounts_last_12m"]
    agg = cur.group_by("customer_id").agg(
        bc_n_reports=pl.len(),
        bc_n_scored=pl.col("score").is_not_null().sum(),
        bc_score_mean=pl.col("score").mean(), bc_score_min=pl.col("score").min(),
        bc_score_max=pl.col("score").max(), bc_score_std=pl.col("score").std(),
        bc_active_max=pl.col("active_accounts").max(),
        bc_outstanding_max=pl.col("outstanding_balance").max(),
        bc_revlimit_max=pl.col("revolving_credit_limit").max(),
        bc_revbal_max=pl.col("revolving_outstanding_balance").max(),
        bc_util_max=pl.col("utilization").max(), bc_util_mean=pl.col("utilization").mean(),
        bc_inq_max=pl.col("inquiries_last_6m").max(), bc_inq_mean=pl.col("inquiries_last_6m").mean(),
        bc_delinq_max=pl.col("delinquent_accounts_last_12m").max(),
        bc_delinq_sum=pl.col("delinquent_accounts_last_12m").sum(),
        bc_stale_mean=pl.col("stale_d").mean(), bc_stale_max=pl.col("stale_d").max(),
        bc_delay_mean=pl.col("delay_d").mean(),
    )
    piv = cur.pivot(on="prov", index="customer_id", values=["score", "utilization", "inquiries_last_6m",
                                                            "delinquent_accounts_last_12m", "outstanding_balance"],
                    aggregate_function="first")
    piv = piv.rename({c: "bp_" + c.replace("inquiries_last_6m", "inq").replace("delinquent_accounts_last_12m", "delinq")
                      .replace("outstanding_balance", "out").replace("utilization", "util")
                      for c in piv.columns if c != "customer_id"})
    # history: bureau pulled at earlier (previous) applications
    hist = b.filter(~pl.col("is_sel")).group_by("customer_id").agg(
        bh_n_pulls=pl.col("application_id").n_unique(),
        bh_score_mean=pl.col("score").mean(), bh_score_min=pl.col("score").min(),
        bh_delinq_max=pl.col("delinquent_accounts_last_12m").max(),
        bh_inq_mean=pl.col("inquiries_last_6m").mean(),
        bh_last_score=pl.col("score").sort_by("report_datetime", "bureau_observation_id").drop_nulls().last(),
        bh_last_outstanding=pl.col("outstanding_balance").sort_by("report_datetime", "bureau_observation_id").last(),
        bh_pulls_180d=pl.col("application_id").filter(pl.col("stale_d") <= 180).n_unique(),
    )
    out = agg.join(piv, on="customer_id", how="left").join(hist, on="customer_id", how="left")
    return out.with_columns(
        bh_score_delta=pl.col("bc_score_mean") - pl.col("bh_last_score"),
        bh_outstanding_delta=pl.col("bc_outstanding_max") - pl.col("bh_last_outstanding"),
    )


def repayment_features(apps: pl.DataFrame, rep: pl.DataFrame, prev: pl.DataFrame) -> pl.DataFrame:
    cut = apps.select("customer_id", cutoff=pl.col("decision_datetime"))
    r = rep.join(cut, on="customer_id").filter(pl.col("due_date") <= pl.col("cutoff"))
    r = r.with_columns(
        age_d=_days("cutoff", "due_date"),
        unpaid=pl.col("amount_due") - pl.col("amount_paid"),
        late_d=pl.when(pl.col("amount_paid") >= pl.col("amount_due")).then(_days("last_payment_datetime", "due_date")),
    )
    w = lambda days: pl.col("age_d") <= days  # noqa: E731
    agg = r.group_by("customer_id").agg(
        rp_n_loans=pl.col("loan_id").n_unique(), rp_n_inst=pl.len(),
        rp_maxdpd_max=pl.col("max_days_past_due").max(), rp_maxdpd_mean=pl.col("max_days_past_due").mean(),
        rp_n_dpd1=(pl.col("max_days_past_due") > 0).sum(), rp_n_dpd30=(pl.col("max_days_past_due") >= 30).sum(),
        rp_n_dpd60=(pl.col("max_days_past_due") >= 60).sum(), rp_n_dpd90=(pl.col("max_days_past_due") >= 90).sum(),
        rp_frac_dpd1=(pl.col("max_days_past_due") > 0).mean(), rp_frac_dpd30=(pl.col("max_days_past_due") >= 30).mean(),
        rp_cur_dpd_max=pl.col("days_past_due").max(), rp_n_unpaid=(pl.col("unpaid") > 0).sum(),
        rp_unpaid_amt=pl.col("unpaid").sum(),
        rp_paid_ratio=pl.col("amount_paid").sum() / pl.col("amount_due").sum(),
        rp_late_mean=pl.col("late_d").mean(), rp_late_max=pl.col("late_d").max(),
        rp_maxdpd_6m=pl.col("max_days_past_due").filter(w(183)).max(),
        rp_maxdpd_12m=pl.col("max_days_past_due").filter(w(365)).max(),
        rp_frac_dpd1_6m=(pl.col("max_days_past_due") > 0).filter(w(183)).mean(),
        rp_late_mean_6m=pl.col("late_d").filter(w(183)).mean(),
        rp_maxdpd_last3=pl.col("max_days_past_due").sort_by("due_date", "loan_id").tail(3).max(),
        rp_days_since_delinq=pl.col("age_d").filter(pl.col("max_days_past_due") > 0).min(),
        rp_days_since_due=pl.col("age_d").min(),
    )
    # active loans: fewer installments observed than contracted term -> remaining obligation
    loans = r.group_by("loan_id").agg(customer_id=pl.col("customer_id").first(), n_seen=pl.len(),
                                      inst=pl.col("amount_due").first())
    loans = loans.join(prev.select("loan_id", "term_months"), on="loan_id", how="left").with_columns(
        remaining=(pl.col("term_months") - pl.col("n_seen")).clip(lower_bound=0))
    act = loans.group_by("customer_id").agg(
        rp_n_active=(pl.col("remaining") > 0).sum(),
        rp_active_inst=pl.col("inst").filter(pl.col("remaining") > 0).sum(),
        rp_remaining_oblig=(pl.col("remaining") * pl.col("inst")).sum(),
    )
    return agg.join(act, on="customer_id", how="left")


def account_features(apps: pl.DataFrame, acc: pl.DataFrame) -> pl.DataFrame:
    cut = apps.select("customer_id", cutoff=pl.col("decision_datetime"))
    a = acc.join(cut, on="customer_id").filter(
        pl.col("month").dt.offset_by("1mo").cast(pl.Datetime) <= pl.col("cutoff")).sort("customer_id", "month")
    a = a.with_columns(mago=pl.col("month").rank("dense", descending=True).over("customer_id"))
    l3 = pl.col("mago") <= 3
    out = a.group_by("customer_id").agg(
        ac_n_months=pl.len(),
        ac_inflow_mean=pl.col("inflows").mean(), ac_outflow_mean=pl.col("outflows").mean(),
        ac_income_mean=pl.col("income_like_credits").mean(), ac_income_std=pl.col("income_like_credits").std(),
        ac_income_min=pl.col("income_like_credits").min(),
        ac_avgbal_mean=pl.col("average_balance").mean(), ac_minbal_min=pl.col("minimum_balance").min(),
        ac_minbal_mean=pl.col("minimum_balance").mean(),
        ac_lowdays_mean=pl.col("low_balance_days").mean(), ac_lowdays_max=pl.col("low_balance_days").max(),
        ac_frac_zero_min=(pl.col("minimum_balance") == 0).mean(),
        ac_net_flow_mean=(pl.col("inflows") - pl.col("outflows")).mean(),
        ac_out_in_ratio=pl.col("outflows").sum() / pl.col("inflows").sum(),
        ac_income_l3=pl.col("income_like_credits").filter(l3).mean(),
        ac_avgbal_l3=pl.col("average_balance").filter(l3).mean(),
        ac_lowdays_l3=pl.col("low_balance_days").filter(l3).mean(),
        ac_out_in_l3=pl.col("outflows").filter(l3).sum() / pl.col("inflows").filter(l3).sum(),
        ac_last_close=pl.col("closing_balance").last(),
        ac_out_in_b=pl.col("outflows").filter(pl.col("mago").is_between(7, 12)).sum()
        / pl.col("inflows").filter(pl.col("mago").is_between(7, 12)).sum(),
        ac_lowdays_b=pl.col("low_balance_days").filter(pl.col("mago").is_between(7, 12)).mean(),
        ac_avgbal_b=pl.col("average_balance").filter(pl.col("mago").is_between(7, 12)).mean(),
        ac_out_in_l1=(pl.col("outflows") / pl.col("inflows")).filter(pl.col("mago") == 1).first(),
        ac_close_slope=pl.cov(pl.col("mago").cast(pl.Float64), pl.col("closing_balance").cast(pl.Float64))
        / pl.col("mago").cast(pl.Float64).var() * -1,
    )
    return out.with_columns(
        ac_income_cv=pl.col("ac_income_std") / pl.col("ac_income_mean"),
        ac_income_trend=pl.col("ac_income_l3") / pl.col("ac_income_mean"),
        ac_bal_trend=pl.col("ac_avgbal_l3") / (pl.col("ac_avgbal_mean") + 1),
        ac_out_in_delta=pl.col("ac_out_in_l3") - pl.col("ac_out_in_b"),
        ac_lowdays_delta=pl.col("ac_lowdays_l3") - pl.col("ac_lowdays_b"),
        ac_bal_delta=(pl.col("ac_avgbal_l3") + 1) / (pl.col("ac_avgbal_b") + 1),
    )


def ecommerce_features(apps: pl.DataFrame, ec: pl.DataFrame) -> pl.DataFrame:
    cut = apps.select("customer_id", cutoff=pl.col("decision_datetime"))
    e = ec.join(cut, on="customer_id").filter(pl.col("transaction_datetime") <= pl.col("cutoff"))
    e = e.with_columns(age_d=_days("cutoff", "transaction_datetime"), done=pl.col("order_status") == "completed")
    # order level: order_total repeats on each item -> dedupe before aggregating
    o = e.group_by("order_id").agg(pl.col("customer_id", "age_d", "order_total", "payment_method", "done").first())
    agg = o.group_by("customer_id").agg(
        ec_n_orders=pl.len(), ec_cancel_rate=1 - pl.col("done").mean(),
        ec_spend=pl.col("order_total").filter("done").sum(),
        ec_aov=pl.col("order_total").filter("done").mean(),
        ec_recency_d=pl.col("age_d").min(), ec_tenure_d=pl.col("age_d").max(),
        ec_orders_7d=(pl.col("age_d") <= 7).sum(), ec_orders_14d=(pl.col("age_d") <= 14).sum(),
        ec_orders_30d=(pl.col("age_d") <= 30).sum(), ec_orders_60d=(pl.col("age_d") <= 60).sum(),
        ec_orders_90d=(pl.col("age_d") <= 90).sum(),
        ec_base_wk=((pl.col("age_d") > 120) & (pl.col("age_d") <= 365)).sum()
        / ((pl.col("age_d").max().clip(upper_bound=365) - 120).clip(lower_bound=7) / 7),
        ec_spend_30d=pl.col("order_total").filter(pl.col("done") & (pl.col("age_d") <= 30)).sum(),
        ec_spend_90d=pl.col("order_total").filter(pl.col("done") & (pl.col("age_d") <= 90)).sum(),
        ec_cancel_90d=(~pl.col("done")).filter(pl.col("age_d") <= 90).sum(),
        **{f"ec_pm_{m}": (pl.col("payment_method") == m).mean()
           for m in ["COD", "e_wallet", "bank_transfer", "debit_card", "credit_card"]},
    )
    cat = (e.filter("done").group_by("customer_id", "product_category").agg(s=pl.col("item_total").sum())
           .with_columns(s=pl.col("s") / pl.col("s").sum().over("customer_id"))
           .pivot(on="product_category", index="customer_id", values="s"))
    cat = cat.rename({c: f"ec_cat_{c}" for c in cat.columns if c != "customer_id"})
    out = agg.join(cat, on="customer_id", how="left").with_columns(
        ec_spend_per_month=pl.col("ec_spend") / (pl.col("ec_tenure_d") / 30.4 + 1),
        ec_spend_90d_ratio=pl.col("ec_spend_90d") / 3 / (pl.col("ec_spend") / (pl.col("ec_tenure_d") / 30.4 + 1) + 1),
    )
    return out.with_columns(
        pl.col("^ec_cat_.*$").fill_null(0),
        ec_accel_30d=(pl.col("ec_orders_30d") / (30 / 7)) / (pl.col("ec_base_wk") + 0.1),
        ec_accel_90d=(pl.col("ec_orders_90d") / (90 / 7)) / (pl.col("ec_base_wk") + 0.1),
    )


def prev_features(apps: pl.DataFrame, prev: pl.DataFrame) -> pl.DataFrame:
    cut = apps.select("customer_id", cutoff=pl.col("decision_datetime"), cur_rate=pl.col("annual_interest_rate"),
                      cur_amt=pl.col("loan_amount"))
    p = prev.join(cut, on="customer_id").filter(pl.col("decision_datetime") <= pl.col("cutoff"))
    p = p.with_columns(age_d=_days("cutoff", "application_datetime"))
    out = p.group_by("customer_id").agg(
        pv_n=pl.len(), pv_n_appr=(pl.col("decision") == "approved").sum(),
        pv_n_rej=(pl.col("decision") == "rejected").sum(), pv_n_canc=(pl.col("decision") == "cancelled").sum(),
        pv_rej_rate=(pl.col("decision") == "rejected").mean(),
        pv_days_since_last=pl.col("age_d").min(), pv_days_since_first=pl.col("age_d").max(),
        pv_n_90d=(pl.col("age_d") <= 90).sum(), pv_n_180d=(pl.col("age_d") <= 180).sum(),
        pv_rej_365d=((pl.col("decision") == "rejected") & (pl.col("age_d") <= 365)).sum(),
        pv_rate_mean=pl.col("annual_interest_rate").mean(), pv_rate_max=pl.col("annual_interest_rate").max(),
        pv_rate_last=pl.col("annual_interest_rate").sort_by("application_datetime", "application_id").drop_nulls().last(),
        pv_appr_ratio=(pl.col("approved_amount") / pl.col("requested_amount")).mean(),
        pv_req_mean=pl.col("requested_amount").mean(), pv_appr_sum=pl.col("approved_amount").sum(),
        pv_appr_max=pl.col("approved_amount").max(),
        pv_last_decision_rej=(pl.col("decision").sort_by("application_datetime", "application_id").last() == "rejected"),
        cur_rate=pl.col("cur_rate").first(), cur_amt=pl.col("cur_amt").first(),
    )
    return out.with_columns(
        pv_rate_delta=pl.col("cur_rate") - pl.col("pv_rate_last"),
        pv_amt_vs_max=pl.col("cur_amt") / pl.col("pv_appr_max"),
        pv_last_decision_rej=pl.col("pv_last_decision_rej").cast(pl.Int8),
    ).drop("cur_rate", "cur_amt")


def v2_features(apps: pl.DataFrame, tables: dict) -> pl.DataFrame:
    """v2 additions (eda/21-26): signals left in the v1 residual. Same cutoff rules as the v1 builders."""
    cut = apps.select("customer_id", "application_id", cutoff=pl.col("decision_datetime"),
                      inst_new=pl.col("proposed_monthly_installment"), amt_new=pl.col("loan_amount"))
    # bureau: score trend/range over every pull, newest report, delinquency over all pulls, external inquiries
    b = tables["bureau_observations"].join(cut.select("customer_id", "cutoff", sel=pl.col("application_id")), on="customer_id")
    b = b.filter(pl.col("available_datetime") <= pl.col("cutoff")).with_columns(
        age_d=_days("cutoff", "report_datetime"), is_sel=pl.col("application_id") == pl.col("sel"))
    bz = b.group_by("customer_id").agg(
        bz_trend=pl.cov(pl.col("age_d"), pl.col("score")) / pl.col("age_d").var() * -365,
        bz_range=pl.col("score").max() - pl.col("score").min(),
        bz_newest=pl.col("score").sort_by("report_datetime", "bureau_observation_id").drop_nulls().last(),
        bz_delinq_all_mean=pl.col("delinquent_accounts_last_12m").mean(),
        bz_active_trend=pl.col("active_accounts").filter("is_sel").mean()
        - pl.col("active_accounts").filter(~pl.col("is_sel")).mean(),
        bz_inq_cur=pl.col("inquiries_last_6m").filter("is_sel").mean(),
    )
    p = tables["previous_applications"].join(cut, on="customer_id").filter(pl.col("decision_datetime") <= pl.col("cutoff"))
    own6 = p.filter(_days("cutoff", "application_datetime") <= 183).group_by("customer_id").agg(own6=pl.len())
    bz = bz.join(own6, on="customer_id", how="left").with_columns(
        bz_inq_external=pl.col("bz_inq_cur") - pl.col("own6").fill_null(0)).drop("bz_inq_cur", "own6")
    # previous approvals: tenor mix, recency, recent approved volume, remaining scheduled obligation
    ap = p.filter(pl.col("decision") == "approved").with_columns(
        age_d=_days("cutoff", "decision_datetime"),
        rem=(pl.col("term_months") - (_days("cutoff", "decision_datetime") / 30.44).floor()).clip(lower_bound=0))
    ob = ap.group_by("customer_id").agg(
        ob_term_mean=pl.col("term_months").mean(), ob_days_since_appr=pl.col("age_d").min(),
        ob_amt_90d=pl.col("approved_amount").filter(pl.col("age_d") <= 90).sum(),
        ob_remaining=(pl.col("scheduled_monthly_installment") * pl.col("rem")).sum(),
        amt_new=pl.col("amt_new").first(),
    ).with_columns(ob_amt_vs_last=pl.col("amt_new") / (pl.col("ob_amt_90d") + 1e6)).drop("amt_new")
    # repayment: first installment, latest loan, 1-29 dpd share, recent trend, streaks, severity when late
    r = tables["repayment_history"].join(cut.select("customer_id", "cutoff"), on="customer_id")
    # loan_id breaks due_date ties between concurrent loans -> deterministic order
    r = r.filter(pl.col("due_date") <= pl.col("cutoff")).sort("customer_id", "due_date", "loan_id").with_columns(
        late=(pl.col("max_days_past_due") > 0).cast(pl.Int32),
        paid=pl.col("amount_paid") >= pl.col("amount_due"),
        late_d=_days("last_payment_datetime", "due_date"),
        k=pl.col("due_date").rank("ordinal").over("loan_id"),
        loan_start=pl.col("due_date").min().over("loan_id"))
    # streak of consecutive late installments within one loan (concurrent loans interleave in time)
    r = r.with_columns(run=(pl.col("late") != pl.col("late").shift(1)).fill_null(True).cum_sum().over("loan_id"))
    streak = (r.filter(pl.col("late") == 1).group_by("customer_id", "loan_id", "run").agg(n=pl.len())
              .group_by("customer_id").agg(rq_streak_max=pl.col("n").max()))
    last = (r.filter(pl.col("loan_start") == pl.col("loan_start").max().over("customer_id"))
            .group_by("customer_id").agg(rq_lastloan_late=pl.col("late").mean(), rq_lastloan_maxdpd=pl.col("max_days_past_due").max()))
    dpd, late = pl.col("max_days_past_due"), pl.col("max_days_past_due") > 0
    rq = r.group_by("customer_id").agg(
        rq_first_inst_late=pl.col("late").filter(pl.col("k") == 1).mean(),
        rq_frac_1_29=(late & (dpd < 30)).mean(),
        rq_late_trend=pl.col("late").tail(6).mean() - pl.col("late").head(pl.len() - 6).mean(),
        sv_maxdpd_mean_late=dpd.filter(late).mean(),
        sv_late_d_mean=pl.col("late_d").filter(pl.col("paid") & (pl.col("late_d") > 0)).mean(),
        sv_share30_of_late=(dpd >= 30).sum() / late.sum(),
        sv_unpaid_of_late=(~pl.col("paid") & late).sum() / late.sum(),
    ).join(streak, on="customer_id", how="left").join(last, on="customer_id", how="left")
    # account: non-income inflows, recent outflows, worst free cash flow, new installment vs free cash / min balance
    a = tables["account_behavior"].join(cut.select("customer_id", "cutoff", "inst_new"), on="customer_id")
    a = a.filter(pl.col("month").dt.offset_by("1mo").cast(pl.Datetime) <= pl.col("cutoff")).with_columns(
        l3=pl.col("month").rank("dense", descending=True).over("customer_id") <= 3,
        free=pl.col("inflows") - pl.col("outflows"))
    cf = a.group_by("customer_id").agg(
        cf_nonincome_share=1 - pl.col("income_like_credits").sum() / pl.col("inflows").sum(),
        cf_nonincome_l3=1 - pl.col("income_like_credits").filter("l3").sum() / pl.col("inflows").filter("l3").sum(),
        cf_out_l3=pl.col("outflows").filter("l3").mean(), cf_free_min=pl.col("free").min(),
        cf_free_l3=pl.col("free").filter("l3").mean(), cf_inc_l3=pl.col("income_like_credits").filter("l3").mean(),
        cf_minbal_l3=pl.col("minimum_balance").filter("l3").min(), inst_new=pl.col("inst_new").first(),
    ).with_columns(
        cf_inst_to_free=pl.col("inst_new") / (pl.col("cf_free_l3").clip(lower_bound=0) + 1e5),
        cf_out_to_inc_l3=pl.col("cf_out_l3") / (pl.col("cf_inc_l3") + 1e5),
        cf_inst_to_minbal=pl.col("inst_new") / (pl.col("cf_minbal_l3") + 1e5),
    ).drop("inst_new", "cf_free_l3", "cf_inc_l3")
    # e-commerce: cancellations, night share, basket size, AOV shift, tightest gap, smooth recency-weighted counts
    e = tables["ecommerce_transactions"].join(cut.select("customer_id", "cutoff"), on="customer_id")
    e = e.filter(pl.col("transaction_datetime") <= pl.col("cutoff"))
    o = e.group_by("order_id").agg(pl.col("customer_id", "cutoff", "transaction_datetime", "order_total", "order_status").first(),
                                   n_items=pl.len())
    o = o.with_columns(age_d=_days("cutoff", "transaction_datetime"), done=pl.col("order_status") == "completed",
                       night=pl.col("transaction_datetime").dt.hour() < 5)
    r28, old = pl.col("age_d") <= 28, pl.col("age_d") > 182
    ec = o.group_by("customer_id").agg(
        ec2_base=(old & (pl.col("age_d") <= 730)).sum()
        / ((pl.col("age_d").max().clip(upper_bound=730) - 182).clip(lower_bound=14) / 7),
        ec2_cancel_28=(~pl.col("done")).filter(r28).sum(),
        ec2_night_share_90=pl.col("night").filter(pl.col("age_d") <= 90).mean(),
        ec2_items_per_order_28=pl.col("n_items").filter(r28).mean(),
        ec2_aov_28=pl.col("order_total").filter(pl.col("done") & r28).mean(),
        ec2_aov_old=pl.col("order_total").filter(pl.col("done") & old).mean(),
        ec2_gap_min_28=pl.col("age_d").filter(r28).sort().diff().min(),
        **{f"ec2_ewm{t}": (-pl.col("age_d") / t).exp().sum() for t in (7, 28, 56)},
    ).with_columns(
        ec2_aov_shift=pl.col("ec2_aov_28") / pl.col("ec2_aov_old"),
        ec2_ewm7_rel=pl.col("ec2_ewm7") / (pl.col("ec2_base") + 0.05),
        ec2_ewm28_exc=pl.col("ec2_ewm28") - pl.col("ec2_base") * 4,
        ec2_ewm56_rel=pl.col("ec2_ewm56") / (pl.col("ec2_base") * 8 + 0.05),
    ).drop("ec2_aov_28", "ec2_aov_old", "ec2_ewm7", "ec2_ewm28", "ec2_ewm56")
    out = cut.select("customer_id")
    for part in (bz, ob, rq, cf, ec):
        out = out.join(part, on="customer_id", how="left")
    return out


PP_CATS = ["groceries", "fashion", "electronics", "home", "health", "education", "personal_care"]


def v5_features(apps: pl.DataFrame, tables: dict) -> pl.DataFrame:
    """v5 additions (eda/40-45): unit-price premium within category, basket quantity, decision month.
    Category reference (mean and sd of log unit price, eda/50): an expanding window over item rows of all customers
    (no labels) dated before the first day of the application's decision month, so every application is normalised
    only with prices that were already observed when it was decided."""
    cut = apps.select("customer_id", cutoff=pl.col("decision_datetime"), dm=pl.col("decision_datetime").dt.truncate("1mo"))
    raw = tables["ecommerce_transactions"].with_columns(lu=(pl.col("item_total") / pl.col("quantity")).log(),
                                                        ym=pl.col("transaction_datetime").dt.truncate("1mo"))
    ms = raw.group_by("product_category", "ym").agg(n=pl.len(), s=pl.col("lu").sum(), ss=(pl.col("lu") ** 2).sum())
    ref = (cut.select("dm").unique().join(ms, how="cross").filter(pl.col("ym") < pl.col("dm"))
           .group_by("product_category", "dm").agg(pl.col("n", "s", "ss").sum())
           .with_columns(mu=pl.col("s") / pl.col("n"),
                         sd=((pl.col("ss") - pl.col("s") ** 2 / pl.col("n")) / (pl.col("n") - 1)).sqrt()))
    e = raw.join(cut, on="customer_id").filter(pl.col("transaction_datetime") <= pl.col("cutoff"))
    e = e.join(ref.select("product_category", "dm", "mu", "sd"), on=["product_category", "dm"], how="left")
    e = e.with_columns(age_d=_days("cutoff", "transaction_datetime"), done=pl.col("order_status") == "completed",
                       z=(pl.col("lu") - pl.col("mu")) / pl.col("sd"))
    lo, hi = pl.col("z").quantile(0.1), pl.col("z").quantile(0.9)
    pp = e.group_by("customer_id").agg(
        pp_mean=pl.col("z").mean(),
        pp_trim=pl.col("z").filter((pl.col("z") > lo) & (pl.col("z") < hi)).mean(),
        pp_done=pl.col("z").filter("done").mean(), pp_min=pl.col("z").min(), pp_max=pl.col("z").max(),
        pp_30=pl.col("z").filter(pl.col("age_d") <= 30).mean(), pp_90=pl.col("z").filter(pl.col("age_d") <= 90).mean(),
        pp_old=pl.col("z").filter(pl.col("age_d") > 182).mean(),
        pp_qty_item=pl.col("quantity").mean(), pp_qty_item_90=pl.col("quantity").filter(pl.col("age_d") <= 90).mean(),
        **{f"pp_cat_{c}": pl.col("z").filter(pl.col("product_category") == c).mean() for c in PP_CATS},
    )
    month = apps.select("customer_id", dec_month=pl.col("decision_datetime").dt.month())
    return month.join(pp, on="customer_id", how="left")


def build_all(tables: dict) -> pl.DataFrame:
    """tables: dict of polars frames keyed by table name. Returns one row per selected application."""
    tr = tables["train_applications"].with_columns(is_train=pl.lit(1))
    te = tables["test_applications"].with_columns(target=pl.lit(None, pl.Int64), is_train=pl.lit(0))
    apps = pl.concat([tr, te], how="diagonal_relaxed")
    f = apps.select("application_id", "customer_id", "decision_datetime", "target", "is_train")
    for part in [
        app_features(apps, tables["customer_profiles"]),
        bureau_features(apps, tables["bureau_observations"]),
        repayment_features(apps, tables["repayment_history"], tables["previous_applications"]),
        account_features(apps, tables["account_behavior"]),
        ecommerce_features(apps, tables["ecommerce_transactions"]),
        prev_features(apps, tables["previous_applications"]),
        v2_features(apps, tables),
        v5_features(apps, tables),
    ]:
        f = f.join(part, on="customer_id", how="left")
    # cross-table ratios: the real affordability picture
    f = f.with_columns(
        has_history=pl.col("pv_n").is_not_null().cast(pl.Int8),
        dti_verified=pl.col("proposed_monthly_installment") / pl.col("ac_income_mean"),
        dti_total=(pl.col("proposed_monthly_installment") + pl.col("rp_active_inst").fill_null(0))
        / pl.col("declared_monthly_income"),
        income_verif_ratio=pl.col("ac_income_mean") / pl.col("declared_monthly_income"),
        debt_to_income=pl.col("bc_outstanding_max") / pl.col("declared_monthly_income"),
        inst_to_avgbal=pl.col("proposed_monthly_installment") / (pl.col("ac_avgbal_mean") + 1),
        ecspend_to_income=pl.col("ec_spend_per_month") / pl.col("declared_monthly_income"),
    )
    return f

In [ ]:
%%writefile {CFG.CODE_DIR}/nn_prep.py
"""Neural-network input preparation (TabM). Pure numpy/scipy so it runs and is verified locally (eda/32).

Per numeric column: mid-rank ECDF -> standard-normal map fitted on train rows only (rank-preserving, so univariate
KS is unchanged; a value shared by many rows, such as 0 days late, lands in the middle of its tie block, not at
an extreme), clipped to +-5, NaN -> 0 (the median). Columns with an identical missingness pattern share one
missing-indicator, because thin-file customers miss whole tables at once. Constant columns are dropped."""
import numpy as np
from scipy.special import ndtri


def nn_prep(train: np.ndarray, test: np.ndarray):
    """train/test: float arrays (rows x numeric features), NaN/inf allowed. Returns (Xtr, Xte, info), float32."""
    train = np.where(np.isfinite(train), train, np.nan)
    test = np.where(np.isfinite(test), test, np.nan)
    ref = [np.sort(train[~np.isnan(train[:, j]), j]) for j in range(train.shape[1])]
    keep = [j for j, s in enumerate(ref) if len(s) and s[0] < s[-1]]

    def transform(a):
        out = np.zeros((a.shape[0], len(keep)), np.float32)
        for i, j in enumerate(keep):
            s, x = ref[j], a[:, j]
            m = ~np.isnan(x)
            u = (np.searchsorted(s, x[m], side="left") + np.searchsorted(s, x[m], side="right")) / (2 * len(s))
            out[m, i] = np.clip(ndtri(np.clip(u, 0.5 / len(s), 1 - 0.5 / len(s))), -5, 5)
        return out

    miss_tr, miss_te = np.isnan(train[:, keep]), np.isnan(test[:, keep])
    seen, ind_cols = set(), []
    for i in range(miss_tr.shape[1]):
        key = np.packbits(miss_tr[:, i]).tobytes()
        if miss_tr[:, i].any() and key not in seen:
            seen.add(key)
            ind_cols.append(i)
    Xtr = np.hstack([transform(train), miss_tr[:, ind_cols].astype(np.float32)])
    Xte = np.hstack([transform(test), miss_te[:, ind_cols].astype(np.float32)])
    return Xtr, Xte, {"kept": keep, "indicator_of": ind_cols}

In [ ]:
%%writefile {CFG.CODE_DIR}/valid2.py
"""validation_v2: one protocol for local screening (one fold) and full Kaggle runs.

Outer: StratifiedKFold(5, shuffle, seed) per seed; the outer validation fold is only scored, never used for any decision.
Inner: a stratified 1/7 split of the outer training fold picks the iteration / epoch (early stopping). The model is then
refitted on the whole outer training fold with that fixed budget (GBDT: best x REFIT, TabM: best epoch).
Every run writes OOF and test predictions with application_id. Blend weights are chosen later, cross-fitted over the
outer folds (eda/blend2.py), never on the rows they are scored on."""
import gc
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from scipy.stats import ks_2samp
from sklearn.model_selection import StratifiedKFold

ID_COLS = ["application_id", "customer_id", "decision_datetime", "target", "is_train"]
CAT_COLS = ["loan_purpose", "region", "employment_type", "housing_status"]
N_FOLDS, INNER, REFIT = 5, 7, 1.15
SEEDS = [42, 7, 2026]
N_ROUNDS, EARLY_STOP = 10000, 200
THREADS = os.cpu_count() or 4

LGB = dict(objective="binary", metric="auc", learning_rate=0.02, num_leaves=31, min_child_samples=100,
           feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, verbose=-1, n_jobs=THREADS)
LGB_S = dict(LGB, num_leaves=7, max_depth=3, min_child_samples=200, feature_fraction=0.4, lambda_l2=10.0)
XGB = dict(objective="binary:logistic", eval_metric="auc", learning_rate=0.02, max_depth=5, min_child_weight=5,
           subsample=0.8, colsample_bytree=0.5, reg_lambda=5.0, tree_method="hist", device="cuda")
XGB_S = dict(XGB, max_depth=3, min_child_weight=20, colsample_bytree=0.4, reg_lambda=10.0)
CAT = dict(learning_rate=0.03, depth=6, l2_leaf_reg=5, loss_function="Logloss", eval_metric="AUC",
           task_type="GPU", verbose=0)
CAT_S = dict(CAT, depth=4, l2_leaf_reg=10)
TABM = dict(lr=2e-3, wd=3e-4, bins=48, batch=512, epochs=60, patience=8)


def ks(y, p):
    return float(ks_2samp(p[y == 1], p[y == 0]).statistic)


def outer(y, seed):
    return list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))


def inner(y_tr, seed, fold):
    """(fit, stop) positions inside the outer training fold."""
    return next(StratifiedKFold(INNER, shuffle=True, random_state=1000 * seed + fold).split(np.zeros(len(y_tr)), y_tr))


def frames(features: pl.DataFrame):
    """features (build_all output) -> X, Xt (pandas; categoricals typed with train categories), y, train ids, test ids."""
    f = features.sort("application_id")
    tr, te = f.filter(pl.col("is_train") == 1), f.filter(pl.col("is_train") == 0)
    cols = sorted(c for c in f.columns if c not in ID_COLS)
    X, Xt = tr.select(cols).to_pandas(), te.select(cols).to_pandas()
    for c in cols:
        if c in CAT_COLS or f[c].dtype == pl.String:
            cats = sorted(X[c].dropna().unique())
            X[c], Xt[c] = pd.Categorical(X[c], categories=cats), pd.Categorical(Xt[c], categories=cats)
        else:
            X[c] = X[c].astype(float).replace([np.inf, -np.inf], np.nan)
            Xt[c] = Xt[c].astype(float).replace([np.inf, -np.inf], np.nan)
    return X, Xt, tr["target"].to_numpy().astype(int), tr["application_id"], te["application_id"]


# --- model fits: fit(X, y, tr, fit, stop, rest, seed) -> list of predictions for each frame in rest ---------------

def lgb_fit(params):
    import lightgbm as lgb

    def fit(X, y, tr, fi, st, rest, seed):
        p = dict(params, seed=seed)
        d_fit = lgb.Dataset(X.iloc[tr[fi]], y[tr[fi]])
        d_st = lgb.Dataset(X.iloc[tr[st]], y[tr[st]], reference=d_fit)
        m = lgb.train(p, d_fit, N_ROUNDS, valid_sets=[d_st], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
        n = math.ceil(m.best_iteration * REFIT)
        m = lgb.train(p, lgb.Dataset(X.iloc[tr], y[tr]), n)
        return [m.predict(r) for r in rest], n
    return fit


def xgb_fit(params):
    import xgboost as xgb

    def fit(X, y, tr, fi, st, rest, seed):
        p = dict(params, seed=seed)
        d_fit = xgb.DMatrix(X.iloc[tr[fi]], y[tr[fi]], enable_categorical=True)
        d_st = xgb.DMatrix(X.iloc[tr[st]], y[tr[st]], enable_categorical=True)
        m = xgb.train(p, d_fit, N_ROUNDS, evals=[(d_st, "st")], early_stopping_rounds=EARLY_STOP, verbose_eval=False)
        n = math.ceil((m.best_iteration + 1) * REFIT)
        m = xgb.train(p, xgb.DMatrix(X.iloc[tr], y[tr], enable_categorical=True), n)
        return [m.predict(xgb.DMatrix(r, enable_categorical=True)) for r in rest], n
    return fit


def cat_fit(params):
    from catboost import CatBoostClassifier, Pool

    def s(df):
        out = df.copy()
        for c in CAT_COLS:
            out[c] = out[c].astype(str)
        return out

    def fit(X, y, tr, fi, st, rest, seed):
        cats = [c for c in CAT_COLS if c in X.columns]
        m = CatBoostClassifier(**params, iterations=N_ROUNDS, od_type="Iter", od_wait=EARLY_STOP, use_best_model=True,
                               random_seed=seed)
        m.fit(Pool(s(X.iloc[tr[fi]]), y[tr[fi]], cat_features=cats), eval_set=Pool(s(X.iloc[tr[st]]), y[tr[st]], cat_features=cats))
        n = math.ceil((m.get_best_iteration() + 1) * REFIT)
        m = CatBoostClassifier(**params, iterations=n, random_seed=seed)
        m.fit(Pool(s(X.iloc[tr]), y[tr], cat_features=cats))
        return [m.predict_proba(s(r))[:, 1] for r in rest], n
    return fit


def tabm_fit(cfg=TABM, device="cuda"):
    import torch
    import tabm
    import rtdl_num_embeddings
    from sklearn.metrics import roc_auc_score
    from nn_prep import nn_prep

    def arrays(X, rows_list):
        nums = [c for c in X.columns if c not in CAT_COLS]
        return [X.iloc[r][nums].to_numpy(dtype=float) for r in rows_list]

    def codes(df):
        return np.column_stack([df[c].cat.codes.to_numpy() + 1 for c in CAT_COLS]).astype(np.int64)

    def train(xn, xc, yv, cards, n_epochs, seed, watch=None):
        torch.manual_seed(seed)
        dev = torch.device(device)
        xn_t, xc_t = torch.as_tensor(xn, device=dev), torch.as_tensor(xc, device=dev)
        yt = torch.as_tensor(yv, dtype=torch.float32, device=dev)
        emb = rtdl_num_embeddings.PiecewiseLinearEmbeddings(
            rtdl_num_embeddings.compute_bins(xn_t, n_bins=cfg["bins"]), d_embedding=16, activation=False, version="B")
        model = tabm.TabM.make(n_num_features=xn.shape[1], cat_cardinalities=cards, d_out=1, num_embeddings=emb).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])

        def predict(a, c):
            model.eval()
            a, c = torch.as_tensor(a, device=dev), torch.as_tensor(c, device=dev)
            with torch.no_grad():
                return torch.cat([torch.sigmoid(model(a[i:i + 1024], c[i:i + 1024]).squeeze(-1).float()).mean(1)
                                  for i in range(0, len(a), 1024)]).cpu().numpy()

        best, best_ep, wait = -1.0, n_epochs, 0
        for ep in range(1, n_epochs + 1):
            model.train()
            for idx in torch.randperm(len(xn_t), device=dev).split(cfg["batch"]):
                out = model(xn_t[idx], xc_t[idx]).squeeze(-1)
                loss = torch.nn.functional.binary_cross_entropy_with_logits(out, yt[idx].unsqueeze(1).expand_as(out))
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            if watch is not None:
                auc = roc_auc_score(watch[2], predict(watch[0], watch[1]))
                best, best_ep, wait = (auc, ep, 0) if auc > best else (best, best_ep, wait + 1)
                if wait >= cfg["patience"]:
                    break
        return model, predict, best_ep

    def fit(X, y, tr, fi, st, rest, seed):
        cards = [len(X[c].cat.categories) + 1 for c in CAT_COLS]
        a_fit, a_st = arrays(X, [tr[fi], tr[st]])
        x_fit, x_st, _ = nn_prep(a_fit, a_st)
        _, _, ep = train(x_fit, codes(X.iloc[tr[fi]]), y[tr[fi]], cards, cfg["epochs"], seed,
                         watch=(x_st, codes(X.iloc[tr[st]]), y[tr[st]]))
        nums = [c for c in X.columns if c not in CAT_COLS]
        a_tr = X.iloc[tr][nums].to_numpy(dtype=float)
        a_rest = np.vstack([r[nums].to_numpy(dtype=float) for r in rest])
        x_tr, x_rest, _ = nn_prep(a_tr, a_rest)
        model, predict, _ = train(x_tr, codes(X.iloc[tr]), y[tr], cards, ep, seed)
        p = predict(x_rest, np.vstack([codes(r) for r in rest]))
        del model
        gc.collect()
        torch.cuda.empty_cache()
        return np.split(p, np.cumsum([len(r) for r in rest])[:-1]), ep
    return fit


def ks_nan(y, x):
    x = np.asarray(x, float)
    if np.isnan(x).all():
        return 0.0
    lo, hi = np.nanmin(x) - 1, np.nanmax(x) + 1
    return max(ks(y, np.where(np.isnan(x), lo, x)), ks(y, np.where(np.isnan(x), hi, x)))


def fm_select(X, y, tr, n_num=100):
    """v8 recipe: numeric columns ranked by univariate KS on the training fold, greedy |Spearman| < 0.95, top n_num,
    plus the 4 categoricals."""
    nums = [c for c in X.columns if c not in CAT_COLS]
    sub = X.iloc[tr]
    ranked = sorted(nums, key=lambda c: -ks_nan(y[tr], sub[c].to_numpy(dtype=float)))[:2 * n_num]
    corr = sub[ranked].rank().corr().abs().fillna(0.0)
    keep = []
    for c in ranked:
        if all(corr.loc[c, k] < 0.95 for k in keep):
            keep.append(c)
        if len(keep) == n_num:
            break
    return keep + CAT_COLS


def icl_fit(make, n_num=100, max_context=60000, chunk=10000, as_codes=False):
    """In-context learners: no early stopping (the inner split is unused). make(seed, cat_idx) -> sklearn-like model."""
    def fit(X, y, tr, fi, st, rest, seed):
        cols = fm_select(X, y, tr, n_num)
        ctx = tr if len(tr) <= max_context else np.random.default_rng(seed).choice(tr, max_context, replace=False)

        def frame(d):
            d = d[cols].reset_index(drop=True)
            if as_codes:
                d = d.copy()
                for c in CAT_COLS:
                    d[c] = d[c].cat.codes.astype(float).replace(-1, np.nan)
                return d.to_numpy(dtype=np.float32)
            return d
        m = make(seed, [cols.index(c) for c in CAT_COLS])
        m.fit(frame(X.iloc[ctx]), y[ctx])
        preds = [np.concatenate([np.asarray(m.predict_proba(frame(r.iloc[i:i + chunk])))[:, 1]
                                 for i in range(0, len(r), chunk)]) for r in rest]
        import torch
        del m
        gc.collect()
        torch.cuda.empty_cache()
        return preds, len(cols)
    return fit


def seg_fit(base):
    """Fit `base` separately on thin-file (has_history=0) and history rows: each segment gets its own context and its
    own fm_select, so thin-file queries are not given ~half all-NaN history columns."""
    def fit(X, y, tr, fi, st, rest, seed):
        h = X["has_history"].to_numpy()
        out = [np.zeros(len(r)) for r in rest]
        for s in (0, 1):
            ts = tr[h[tr] == s]
            masks = [r["has_history"].to_numpy() == s for r in rest]
            preds, n = base(X, y, ts, fi, st, [r[m] for r, m in zip(rest, masks)], seed)
            for o, p_, m in zip(out, preds, masks):
                o[m] = p_
        return out, n
    return fit


def causilo_fit(n_num=100, n_est=4, max_context=60000, device="cuda"):
    from causilo import CausiloClassifier
    return icl_fit(lambda seed, ci: CausiloClassifier(n_estimators=n_est, random_state=seed, device=device),
                   n_num, max_context)


def tabpfn_fit(n_num=100, n_est=4, max_context=60000, device="cuda"):
    from tabpfn import TabPFNClassifier
    return icl_fit(lambda seed, ci: TabPFNClassifier(n_estimators=n_est, random_state=seed, device=device,
                                                     ignore_pretraining_limits=True, categorical_features_indices=ci,
                                                     fit_mode="fit_with_cache"),
                   n_num, max_context, chunk=5000, as_codes=True)


def tabdpt_fit(n_num=100, context=4096, n_ens=8, device="cuda"):
    """TabDPT retrieves `context` nearest training rows per query (its own context reduction)."""
    from tabdpt import TabDPTClassifier

    class M:
        def __init__(self, seed):
            self.m, self.seed = TabDPTClassifier(device=device, context_reduction="retrieval", missing_indicators=True,
                                                 compile=False, verbose=False), seed

        def fit(self, Xa, ya):
            self.m.fit(Xa, ya)

        def predict_proba(self, Xa):
            return np.mean([self.m.predict_proba(Xa, context_size=context, batch_size=64, seed=self.seed + k) for k in range(n_ens)], axis=0)
    return icl_fit(lambda seed, ci: M(seed), n_num, 10 ** 9, chunk=2048, as_codes=True)


def kumo_fit(size="large", n_num=100, max_context=20000, n_est=4, chunk=1000, n_bag=1, g_off=0):
    """NVIDIA Kumo-Tabular (structured-data-models) in-context learner on fm_select features (categoricals as integer
    codes, numeric-only model). Context = stratified subsample of `max_context` training-fold rows; no early stopping."""
    import torch
    import sdm

    def frame(d, cols):
        d = d[cols].copy()
        for c in CAT_COLS:
            d[c] = d[c].cat.codes.astype(float).replace(-1, np.nan)
        return d.astype(float).reset_index(drop=True)

    def fit(X, y, tr, fi, st, rest, seed):
        """n_bag context draws (stratified, independent) averaged: one 20k draw moves fold KS by up to ~0.01."""
        cols = fm_select(X, y, tr, n_num)
        dev = torch.device("cuda")
        model = sdm.models.KumoTabular(task="classification", size=size, device=dev)
        q = pd.concat([frame(r, cols) for r in rest], ignore_index=True)
        p = np.mean([one(model, X, y, tr, cols, q, seed * 100 + b, dev, g_off) for b in range(n_bag)], axis=0)
        del model
        gc.collect()
        torch.cuda.empty_cache()
        return np.split(p, np.cumsum([len(r) for r in rest])[:-1]), len(tr) if len(tr) <= max_context else max_context

    def one(model, X, y, tr, cols, q, seed, dev, g_off=0):
        rng = np.random.default_rng(seed)
        ctx = tr if len(tr) <= max_context else np.sort(np.concatenate(
            [rng.choice(tr[y[tr] == c], int(round(max_context * (y[tr] == c).mean())), replace=False) for c in (0, 1)]))
        cdf = frame(X.iloc[ctx], cols)
        full = pd.concat([cdf.assign(target=y[ctx].astype(str)), q.assign(target="0")], ignore_index=True)
        stypes = sdm.infer_stypes(full, overrides={"target": "categorical", **{c: "numerical" for c in cols}})
        t = sdm.TableTensor.from_pandas(df=full, stypes=stypes, device=dev)
        n = len(cdf)
        xc, yc = t[:n].drop_columns("target"), t[:n, "target"]
        g = torch.Generator(device=dev).manual_seed(seed + 1000 * g_off)  # g_off: independent estimator draw, same context
        out = []
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
            for i in range(n, len(full), chunk):
                r = model(x_context=xc, y_context=yc, x_query=t[i:min(i + chunk, len(full))].drop_columns("target"),
                          num_estimators=n_est, generator=g)
                out.append(r.to_pandas()["1"].to_numpy())
        del t
        return np.concatenate(out)
    return fit


def realmlp_fit(n_ens=1, **kw):
    """pytabkit RealMLP-TD (tuned defaults). Checkpoint selection only on the inner stop split (X_val), then the
    selected network predicts the outer fold and test (no refit: RealMLP's schedule is tied to its epoch budget)."""
    from pytabkit import RealMLP_TD_Classifier

    def prep(X, rows):
        """Median fill fitted on `rows`, one indicator per distinct missingness pattern (thin-file blocks)."""
        nums = [c for c in X.columns if c not in CAT_COLS]
        med = X.iloc[rows][nums].median()
        miss = X.iloc[rows][nums].isna()
        groups, seen = [], set()
        for c in nums:
            key = np.packbits(miss[c].to_numpy()).tobytes()
            if miss[c].any() and key not in seen:
                seen.add(key)
                groups.append(c)

        def tf(d):
            out = d[nums].fillna(med).fillna(0.0)
            for c in groups:
                out[f"na_{c}"] = d[c].isna().astype(float)
            for c in CAT_COLS:
                out[c] = d[c]
            return out
        return tf

    def fit(X, y, tr, fi, st, rest, seed):
        tf = prep(X, tr[fi])
        X = tf(X)
        rest = [tf(r) for r in rest]
        m = RealMLP_TD_Classifier(device="cuda", random_state=seed, n_ens=n_ens, verbosity=0, val_metric_name="1-auc_ovr",
                                  **kw)
        m.fit(X.iloc[tr[fi]], y[tr[fi]], X_val=X.iloc[tr[st]], y_val=y[tr[st]], cat_col_names=CAT_COLS)
        return [m.predict_proba(r)[:, 1] for r in rest], 0
    return fit


def woe_lr_fit(n_bins=20, alpha=20.0, Cs=(0.003, 0.01, 0.03, 0.1, 0.3)):
    """Credit scorecard: per feature, bins fitted on the training rows (NaN and exact zero get their own bins, else
    quantile bins of the rest; categoricals by level), smoothed WOE = log odds of the bin shrunk to the prior with
    `alpha` pseudo-rows, then L2 logistic regression on the standardised WOE columns. C is picked on the inner stop
    split, then the LR is refitted on the whole training fold (binning refitted there too)."""
    from sklearn.linear_model import LogisticRegression

    def encoder(X, rows, yv):
        prior = yv.mean()
        lo = np.log(prior / (1 - prior))
        specs = []
        for c in X.columns:
            if c in CAT_COLS:
                codes = X[c].cat.codes.to_numpy()
                specs.append((c, "cat", None))
                b = codes[rows]
            else:
                x = X[c].to_numpy(dtype=float)[rows]
                nz = np.isfinite(x) & (x != 0)
                edges = np.unique(np.quantile(x[nz], np.linspace(0, 1, n_bins + 1))[1:-1]) if nz.sum() > n_bins else np.array([])
                specs.append((c, "num", edges))
                b = bins_of(x, edges)
            keys, inv = np.unique(b, return_inverse=True)
            pos = np.bincount(inv, weights=yv, minlength=len(keys))
            n = np.bincount(inv, minlength=len(keys))
            rate = (pos + alpha * prior) / (n + alpha)
            specs[-1] = specs[-1] + (dict(zip(keys.tolist(), (np.log(rate / (1 - rate)) - lo).tolist())),)
        return specs

    def bins_of(x, edges):
        out = np.where(np.isnan(x), -2, np.where(x == 0, -1, np.searchsorted(edges, x, side="right")))
        return out

    def transform(specs, D):
        cols = []
        for c, kind, edges, table in specs:
            b = D[c].cat.codes.to_numpy() if kind == "cat" else bins_of(D[c].to_numpy(dtype=float), edges)
            cols.append(np.array([table.get(int(k), 0.0) for k in b]) if len(table) < 50 else
                        np.vectorize(lambda k: table.get(int(k), 0.0))(b))
        return np.column_stack(cols)

    def fit_lr(A, yv, C, seed):
        mu, sd = A.mean(0), A.std(0) + 1e-9
        m = LogisticRegression(C=C, max_iter=2000, random_state=seed).fit((A - mu) / sd, yv)
        return lambda B: m.decision_function((B - mu) / sd)

    def fit(X, y, tr, fi, st, rest, seed):
        sp = encoder(X, tr[fi], y[tr[fi]])
        A, S = transform(sp, X.iloc[tr[fi]]), transform(sp, X.iloc[tr[st]])
        best = max(Cs, key=lambda C: ks(y[tr[st]], fit_lr(A, y[tr[fi]], C, seed)(S)))
        sp = encoder(X, tr, y[tr])
        f = fit_lr(transform(sp, X.iloc[tr]), y[tr], best, seed)
        return [1 / (1 + np.exp(-f(transform(sp, r)))) for r in rest], best
    return fit


def lr_q_fit(Cs=(0.001, 0.003, 0.01, 0.03, 0.1, 1.0), onehot=True):
    """Logistic regression on nn_prep inputs (mid-rank ECDF -> normal fitted on the training rows, NaN -> 0, one
    indicator per missingness pattern) + one-hot categoricals. C picked on the inner stop split, refit on the fold."""
    from sklearn.linear_model import LogisticRegression
    from nn_prep import nn_prep

    def mats(X, fit_rows, others):
        nums = [c for c in X.columns if c not in CAT_COLS]
        a, b, _ = nn_prep(X.iloc[fit_rows][nums].to_numpy(dtype=float),
                          np.vstack([o[nums].to_numpy(dtype=float) for o in others]))
        def oh(d):
            return np.column_stack([(d[c].cat.codes.to_numpy()[:, None] == np.arange(len(d[c].cat.categories))).astype(np.float32)
                                    for c in CAT_COLS])
        if onehot:
            a = np.hstack([a, oh(X.iloc[fit_rows])])
            b = np.hstack([b, np.vstack([oh(o) for o in others])])
        return a, np.split(b, np.cumsum([len(o) for o in others])[:-1])

    def fit(X, y, tr, fi, st, rest, seed):
        a, (s_,) = mats(X, tr[fi], [X.iloc[tr[st]]])
        best = max(Cs, key=lambda C: ks(y[tr[st]], LogisticRegression(C=C, max_iter=3000).fit(a, y[tr[fi]]).decision_function(s_)))
        a, outs = mats(X, tr, rest)
        m = LogisticRegression(C=best, max_iter=3000).fit(a, y[tr])
        return [m.predict_proba(o)[:, 1] for o in outs], best
    return fit


def run(name, fit, X, Xt, y, ids, test_ids, out: Path, seeds=SEEDS, folds=None, deadline=None):
    """Runs fit over seeds x outer folds. OOF = mean over seeds (NaN where no fold ran); test = mean over all fits.
    Writes out/oof_<name>.parquet, test_<name>.parquet and log_<name>.csv."""
    out.mkdir(parents=True, exist_ok=True)
    oof_sum, oof_n, test_sum, log = np.zeros(len(y)), np.zeros(len(y)), np.zeros(len(Xt)), []
    for seed in seeds:
        for k, (tr, va) in enumerate(outer(y, seed)):
            if folds is not None and k not in folds:
                continue
            if deadline is not None and time.time() > deadline:
                raise TimeoutError(f"{name}: deadline before seed {seed} fold {k}")
            t0 = time.time()
            fi, st = inner(y[tr], seed, k)
            (pv, pt), budget = fit(X, y, tr, fi, st, [X.iloc[va], Xt], seed)
            oof_sum[va] += pv
            oof_n[va] += 1
            test_sum += pt
            log.append(dict(model=name, seed=seed, fold=k, ks=ks(y[va], pv), budget=budget, sec=round(time.time() - t0, 1)))
            print(log[-1], flush=True)
            pd.DataFrame(log).to_csv(out / f"log_{name}.csv", index=False)
    oof = np.where(oof_n > 0, oof_sum / np.maximum(oof_n, 1), np.nan)
    pl.DataFrame({"application_id": ids, "target": y, name: oof}).write_parquet(out / f"oof_{name}.parquet")
    pl.DataFrame({"application_id": test_ids, name: test_sum / len(log)}).write_parquet(out / f"test_{name}.parquet")
    m = oof_n > 0
    print(f"{name}: OOF KS {ks(y[m], oof[m]):.5f} on {m.sum()} rows", flush=True)
    return oof, test_sum / len(log), log


def manifest(out: Path, extra=None):
    import importlib.metadata as md
    pk = {}
    for n in ["numpy", "pandas", "polars", "scikit-learn", "scipy", "lightgbm", "xgboost", "catboost", "torch", "tabm",
              "rtdl_num_embeddings", "causilo"]:
        try:
            pk[n] = md.version(n)
        except md.PackageNotFoundError:
            pk[n] = None
    cfg = dict(N_FOLDS=N_FOLDS, INNER=INNER, REFIT=REFIT, SEEDS=SEEDS, N_ROUNDS=N_ROUNDS, EARLY_STOP=EARLY_STOP,
               LGB=LGB, LGB_S=LGB_S, XGB=XGB, XGB_S=XGB_S, CAT=CAT, CAT_S=CAT_S, TABM=TABM)
    (out / "manifest.json").write_text(json.dumps(dict(packages=pk, config=cfg, **(extra or {})), indent=2, default=str))

In [ ]:
%%writefile {CFG.CODE_DIR}/run_valid2.py
"""Driver for validation_v2. Local screening:  python run_valid2.py OUT_DIR lgb,tabm --folds 0 --seeds 42
Kaggle full run: same call without --folds/--seeds. Extra feature parquet (application_id + columns) via --extra."""
import argparse
import time
from pathlib import Path

import polars as pl

import valid2 as v

MODELS = {
    "lgb": lambda: v.lgb_fit(v.LGB), "lgb_s": lambda: v.lgb_fit(v.LGB_S),
    "xgb": lambda: v.xgb_fit(v.XGB), "xgb_s": lambda: v.xgb_fit(v.XGB_S),
    "cat": lambda: v.cat_fit(v.CAT), "cat_s": lambda: v.cat_fit(v.CAT_S),
    "tabm": lambda: v.tabm_fit(),
    "causilo": lambda: v.causilo_fit(),
    "realmlp": lambda: v.realmlp_fit(),
    "woe_lr": lambda: v.woe_lr_fit(),
    "lr_q": lambda: v.lr_q_fit(),
}


def model(name):
    """'causilo_e16_f150' -> causilo_fit(n_est=16, n_num=150); other names from MODELS."""
    if name.startswith("seg_"):
        return v.seg_fit(model(name[4:]))
    base = name.split("_")[0]
    if base in ("causilo", "tabpfn", "tabdpt", "kumo"):
        parts = name.split("_")[1:]
        kw = {"size": parts.pop(0)} if base == "kumo" and parts and parts[0] in ("small", "medium", "large") else {}
        kw |= {dict(e="n_est", f="n_num", c="max_context", x="context", n="n_ens", b="n_bag", g="g_off")[t[0]]: int(t[1:]) for t in parts}
        return getattr(v, f"{base}_fit")(**kw)
    return MODELS[name]()


def main(features: pl.DataFrame, out: Path, names, folds=None, seeds=None, extra=None, drop=(), deadline=None):
    if extra is not None:
        features = features.join(extra, on="application_id", how="left", validate="1:1")
    features = features.drop([c for c in drop if c in features.columns])
    X, Xt, y, ids, tids = v.frames(features)
    out.mkdir(parents=True, exist_ok=True)
    v.manifest(out, dict(models=names, folds=folds, seeds=seeds or v.SEEDS, n_features=X.shape[1],
                         extra=None if extra is None else extra.columns, drop=list(drop)))
    res = {}
    for n in names:
        res[n] = v.run(n, model(n), X, Xt, y, ids, tids, out, seeds=seeds or v.SEEDS, folds=folds, deadline=deadline)
    return res


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("out")
    ap.add_argument("models")
    ap.add_argument("--folds", type=lambda s: [int(x) for x in s.split(",")])
    ap.add_argument("--seeds", type=lambda s: [int(x) for x in s.split(",")])
    ap.add_argument("--extra", help="parquet with application_id + new columns")
    ap.add_argument("--drop", default="", help="comma-separated feature columns to drop")
    ap.add_argument("--features", help="build_all parquet (default: data_pq/features.parquet via common.features)")
    ap.add_argument("--hours", type=float, help="deadline in hours from start")
    a = ap.parse_args()
    t0 = time.time()
    if a.features:
        feats = pl.read_parquet(a.features)
    else:
        from common import features
        feats = features()
    main(feats, Path(a.out), a.models.split(","), a.folds, a.seeds,
         pl.read_parquet(a.extra) if a.extra else None, [c for c in a.drop.split(",") if c],
         t0 + a.hours * 3600 if a.hours else None)
    print(f"done in {time.time() - t0:.0f}s")

In [ ]:
%%writefile {CFG.CODE_DIR}/kumo_fulltrain.py
"""Kumo-Tabular large with the whole training set (70k rows) as context, predicting the 30k test rows once.
Cheaper alternative to averaging the 5 fold models for the final notebook. Prints wall time and peak VRAM.
Usage: python kumo_fulltrain.py OUT_DIR [--features PATH]"""
import argparse
import os
import time
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # v30 v1 OOM: 7 GB reserved-unallocated
import numpy as np
import polars as pl
import torch

import valid2 as v

ap = argparse.ArgumentParser()
ap.add_argument("out")
ap.add_argument("--features", default=None)
ap.add_argument("--chunk", type=int, default=500)
a = ap.parse_args()
if a.features:
    feats = pl.read_parquet(a.features)
else:
    from common import features
    feats = features()
X, Xt, y, ids, tids = v.frames(feats)
t0 = time.time()
(p,), n = v.kumo_fit(size="large", max_context=10 ** 6, chunk=a.chunk)(X, y, np.arange(len(y)), None, None, [Xt], 42)
out = Path(a.out)
out.mkdir(parents=True, exist_ok=True)
pl.DataFrame({"application_id": tids, "kumo_full70k": p}).write_parquet(out / "test_kumo_full70k.parquet")
print(f"context {n}, test rows {len(p)}, {time.time() - t0:.0f} s, peak VRAM {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")

---

# Feature Engineering <a name="4"></a>

---

`build_all` menggabungkan fitur aplikasi, biro, repayment, rekening, e-commerce (termasuk premium harga per kategori dengan referensi expanding per bulan keputusan), dan aplikasi sebelumnya menjadi satu baris per aplikasi. Setiap builder memfilter baris tabel bantu di `decision_datetime`.

In [ ]:
build_all = __import__('features').build_all
t0 = time.time()
feats = build_all(tables)
feats.write_parquet(CFG.RUN_DIR / 'features.parquet')
print(feats.shape, f'{time.time() - t0:.0f} s')
print(feats.group_by('is_train').agg(pl.len(), pl.col('target').mean().alias('bad_rate')))
del tables

#### Insights

> Train dan test memakai builder yang sama dalam satu pass, sehingga tidak ada perbedaan definisi fitur antar split. Label hanya dipakai di langkah model, di dalam train fold.

---

# Model Training <a name="5"></a>

---

Dua rantai proses berjalan paralel, satu per GPU. GPU 0 menjalankan Kumo-Tabular large dengan 70.000 baris train sebagai context. GPU 1 menjalankan enam GBDT (3 seed outer x 5 fold) lalu Causilo (5 fold). Setiap model menulis `oof_<model>.parquet`, `test_<model>.parquet`, dan `log_<model>.csv` ke `run/`.

In [ ]:
def cmd(script, *args):
    return ' '.join(map(repr, [sys.executable, str(CFG.CODE_DIR / script), *args]))

feat_path = str(CFG.RUN_DIR / 'features.parquet')
chains = {
    0: [cmd('kumo_fulltrain.py', str(CFG.RUN_DIR), '--features', feat_path)],
    1: [cmd('run_valid2.py', str(CFG.RUN_DIR / 'gbdt'), ','.join(CFG.TREES), '--seeds', CFG.TREE_SEEDS,
            '--features', feat_path, '--hours', '11'),
        cmd('run_valid2.py', str(CFG.RUN_DIR / 'fm'), 'causilo', '--seeds', CFG.ICL_SEED,
            '--features', feat_path, '--hours', '11')],
}
procs = {}
for gpu, steps in chains.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
    procs[gpu] = subprocess.Popen(' && '.join(steps), shell=True, env=env, cwd=CFG.CODE_DIR,
                                  stdout=open(CFG.RUN_DIR / f'gpu{gpu}.log', 'w'), stderr=subprocess.STDOUT)
t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(120)
for gpu, p in procs.items():
    print(f'GPU {gpu}: exit {p.returncode}, {(time.time() - t0) / 3600:.1f} h')
    print(''.join(open(CFG.RUN_DIR / f'gpu{gpu}.log').readlines()[-6:]))
assert all(p.returncode == 0 for p in procs.values())

Skor KS OOF per model (validation_v2) dibaca dari log. Kumo tidak punya OOF di notebook ini karena dijalankan sekali di seluruh train; skor OOF-nya diukur terpisah saat validasi (0,52665, context penuh per fold).

In [ ]:
logs = pl.concat([pl.read_csv(p) for p in sorted(CFG.RUN_DIR.rglob('log_*.csv'))], how='diagonal')
print(logs.group_by('model').agg(pl.col('ks').mean().alias('mean_fold_ks'), pl.len().alias('fits'),
                                 (pl.col('sec').sum() / 60).alias('minutes')).sort('model'))

---

# Blending and Submission <a name="6"></a>

---

Prediksi test setiap model diubah menjadi rank ternormalisasi lalu dirata-rata dengan bobot tetap: 1/18 untuk setiap GBDT (total 1/3), 1/3 Causilo, dan 1/3 Kumo. Bobot ini ditetapkan sebelum evaluasi akhir dan diuji dengan cross-fit di OOF, bukan dioptimasi di leaderboard.

In [ ]:
test = None
for p in sorted(CFG.RUN_DIR.rglob('test_*.parquet')):
    t = pl.read_parquet(p)
    test = t if test is None else test.join(t, on='application_id', how='inner', validate='1:1')
test = test.sort('application_id')
score = sum(w * rankdata(test[m].to_numpy()) / test.height for m, w in CFG.WEIGHTS.items()) / sum(CFG.WEIGHTS.values())
sub = pl.DataFrame({'application_id': test['application_id'], 'default_probability': score})

ref = pl.read_csv(CFG.DATA_DIR / 'sample_submission.csv')
assert sub.height == ref.height == 30000 and sub['application_id'].n_unique() == 30000
assert set(sub['application_id']) == set(ref['application_id'])
assert np.isfinite(score).all() and score.min() >= 0 and score.max() <= 1
sub.write_csv(CFG.OUTPUT_DIR / 'submission.csv')
print(sub.head(), sub['default_probability'].describe(), sep='\n')

#### Insights

> File `submission.csv` berisi 30.000 baris dengan ID unik yang sama persis dengan `sample_submission.csv` dan skor di [0, 1].

> Karena KS hanya bergantung pada urutan, skor rank-average tidak perlu dikalibrasi menjadi probabilitas.

---

# Conclusion <a name="7"></a>

---

Blend rank enam GBDT, Causilo, dan Kumo-Tabular memberi KS OOF bersih 0,526 (resep blend lama 0,524) dan public leaderboard sekitar 0,563-0,564. Dua pelajaran utama dari validasi: early stopping di fold OOF membuat skor validasi lama terlalu optimis, dan model in-context learning hanya membantu bila dicampur dengan GBDT; blend tanpa GBDT turun tajam di leaderboard walaupun OOF-nya setara.